In [ ]:
from sage.all import *
from Crypto.Util.number import long_to_bytes
from Crypto.Cipher import AES
from Crypto.Util.Padding import unpad

N = 121288600621198389662246479277632294800423697823363188896668775456771641807233781416525282234787873435904747571468452950479817935684848143651716343606633656969395065588423982440884464542428742861388200306417822228591316703916504170245990423925894477848679490979364923848426643149659758241239900845544537886777
c = 3756824985347508967549776773725045773059311839370527149219720084008312247164501688241698562854942756369420003479117
a2_high = 9012778
a1 = 621315
a0 = 452775142
iv_hex = "bf38e64bb5c1b069a07b7d1d046a9010"
ct_hex = "8966006c4724faf53883b56a1a8a08ee17b1535e1657c16b3b129ee2d2e389744c943014eb774cd24a5d0f7ad140276fdec72eb985b6de67b8e4674b0bcdc4a5"

iv = bytes.fromhex(iv_hex.strip())
ct = bytes.fromhex(ct_hex.strip())

LOW_BITS = 16

def solve_direct():
    print("-" * 60)
    print("[*] 分析: c 的位数远小于 N，这是一个整数方程。")
    print("[*] 开始遍历 y (0 ~ 65535)...")

    # 定义整数多项式环
    R = PolynomialRing(ZZ, 'x')
    x = R.gen()

    # 预计算常量
    A_high = a2_high << LOW_BITS
    const_part = a0 - c
    
    # 遍历 y
    # 方程: x^3 + (A_high + y)*x^2 + a1*x + (a0 - c) = 0
    
    for y_val in range(65536):
        # 构造当前 y 对应的系数
        coeff_x2 = A_high + y_val
        
        # 定义多项式
        f = x**3 + coeff_x2 * x**2 + a1 * x + const_part
        
        # 求解整数根
        # 由于是单调函数且求解整数根，Sage 的 .roots() 非常快
        roots = f.roots(ring=ZZ, multiplicities=False)
        
        if roots:
            m_found = roots[0]
            print(f"[+] 找到解! y = {y_val}")
            print(f"[+] m = {m_found}")
            return m_found
            
    return None

# 执行求解
m_val = solve_direct()

if m_val:
    print("-" * 60)
    print("[*] 尝试解密...")
    try:
        # 恢复 Key
        key = long_to_bytes(int(m_val))
        # 补齐到 16 字节
        key = key.rjust(16, b'\0')
        
        cipher = AES.new(key, AES.MODE_CBC, iv=iv)
        pt = unpad(cipher.decrypt(ct), 16)
        
        print(f"[SUCCESS] Flag: {pt.decode()}")
    except Exception as e:
        print(f"[-] 解密失败: {e}")
        # 打印原始 hex 方便调试
        cipher = AES.new(key, AES.MODE_CBC, iv=iv)
        print(f"[-] Raw Decrypted (hex): {cipher.decrypt(ct).hex()}")
else:
    print("[-] 未找到解，请检查参数是否正确。")

------------------------------------------------------------
[*] 分析: c 的位数远小于 N，这是一个整数方程。
[*] 开始遍历 y (0 ~ 65535)...
[+] 找到解! y = 10219
[+] m = 155455820692697783953491152103673430935
------------------------------------------------------------
[*] 尝试解密...
[SUCCESS] Flag: ISCTF{i7_533M5_Lik3_You_R34lLy_UNd3R574nd_Polinomials_4nD_RSA}
